In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV ,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import joblib
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.neighbors import KNeighborsClassifier



In [11]:
!pip install xgboost


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:

x_train=np.load('./data/embeddings/train_embeddings.npy')
x_test=np.load('./data/embeddings/test_embeddings.npy')


In [4]:
x_train

array([[ 0.07367804, -0.02631131,  0.0390068 , ..., -0.07857121,
         0.05455049,  0.07414829],
       [-0.0068568 , -0.06906828, -0.04574034, ..., -0.06036085,
         0.04355335,  0.04123302],
       [-0.01750738, -0.03781661,  0.02444598, ..., -0.11092535,
        -0.02253906,  0.04098497],
       ...,
       [-0.02288846,  0.01629856, -0.03647592, ...,  0.03625312,
         0.02009865,  0.04081287],
       [-0.03029878, -0.02274068, -0.00463907, ...,  0.01157963,
        -0.01234237,  0.02648924],
       [-0.03888904, -0.05788031, -0.02970093, ...,  0.03459368,
         0.02085046, -0.06742197]], shape=(120000, 384), dtype=float32)

In [5]:
metadata_train=pd.read_csv("./data/metadata/train_metadata.csv")
metadata_test=pd.read_csv("./data/metadata/test_metadata.csv")
y_test=metadata_test["label"].values
y_train=metadata_train["label"].values



In [6]:
import numpy as np

print(y_train)


[2 2 2 ... 1 1 1]


In [13]:
models={
    "SVM":LinearSVC(),
    "xgboost":xgb.XGBClassifier(),
    "LogisticRegression":LogisticRegression(),
    "knn" : KNeighborsClassifier()
}

In [14]:
for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred_train=model.predict(x_train)
    y_pred_test=model.predict(x_test)

    joblib.dump(model,f'./models/{name}_model.joblib')
    print("="*50)
    print("="*50)
    print("="*50)

    print(f"=== {name} ===")
    print("="*50)

    print("Model Accuracy:\n")
    print(f"accuracy_score train:\n{accuracy_score(y_train,y_pred_train)}\n")

    print(f"accuracy_score test:\n{accuracy_score(y_test,y_pred_test)}\n")
    print("="*50)
    print("="*50)

    print("Confusion Matrix:\n")

    print(f"confusion_matrix train : \n{confusion_matrix(y_train,y_pred_train)}")
    print("="*50)
    print(f"confusion_matrix test :\n{confusion_matrix(y_test,y_pred_test)}")
    print("="*50)
    print("="*50)

    print("classification report:\n")

    print(f"classification_report train \n:{classification_report(y_train,y_pred_train)}")
    print("="*50)
    print(f"classification_report test \n:{classification_report(y_test,y_pred_test)}")
    print("\n")


=== SVM ===
Model Accuracy:

accuracy_score train:
0.892775

accuracy_score test:
0.89

Confusion Matrix:

confusion_matrix train : 
[[26498   942  1607   953]
 [  430 29123   288   159]
 [ 1102   293 25608  2997]
 [ 1139   216  2741 25904]]
confusion_matrix test :
[[1684   60  100   56]
 [  33 1836   25    6]
 [  72   18 1623  187]
 [  75   17  187 1621]]
classification report:

classification_report train 
:              precision    recall  f1-score   support

           0       0.91      0.88      0.90     30000
           1       0.95      0.97      0.96     30000
           2       0.85      0.85      0.85     30000
           3       0.86      0.86      0.86     30000

    accuracy                           0.89    120000
   macro avg       0.89      0.89      0.89    120000
weighted avg       0.89      0.89      0.89    120000

classification_report test 
:              precision    recall  f1-score   support

           0       0.90      0.89      0.89      1900
           1  

In [ ]:
model = xgb.XGBClassifier(
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=42,        
)


param_grid={
    'n_estimators': [50,100],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.01],
    'subsample': [0.8, 1.0]
}


grid_search= GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    refit=True,
    cv=3,
    n_jobs=-1   
)
grid_search.fit(x_train,y_train)
best_model=grid_search.best_estimator_
y_pred=best_model.predict(x_test)

print(f"accuracy_score:{accuracy_score(y_test,y_pred)}\n")

print("Confusion Matrix:\n")


print(f"confusion_matrix :  {confusion_matrix(y_test,y_pred)}")

print(f"classification_report: { classification_report(y_test,y_pred)}")

print("\n")

joblib.dump(best_model,"./models/best_model.joblib")

accuracy_score:0.8752631578947369

Confusion Matrix:

confusion_matrix :  [[1672   61  110   57]
 [  56 1796   34   14]
 [  84   27 1584  205]
 [  76   31  193 1600]]
classification_report:               precision    recall  f1-score   support

           0       0.89      0.88      0.88      1900
           1       0.94      0.95      0.94      1900
           2       0.82      0.83      0.83      1900
           3       0.85      0.84      0.85      1900

    accuracy                           0.88      7600
   macro avg       0.88      0.88      0.88      7600
weighted avg       0.88      0.88      0.88      7600





['./models/best_model.joblib']

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import joblib

model = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [ 7, 9, 11, 13,15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(x_train, y_train)

best_model = grid_search.best_estimator_

y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)



if hasattr(best_model, "predict_proba"):
    y_prob_train = best_model.predict_proba(x_train )

    print("ROC AUC:",
          roc_auc_score(
              y_train,
              y_prob_train,
              multi_class='ovr'
          )
    )
    
if hasattr(best_model, "predict_proba"):
    y_prob_test = best_model.predict_proba(x_test)

    print("ROC AUC:",
          roc_auc_score(
              y_test,
              y_prob_test,
              multi_class='ovr'
          )
    )




print("Accuracy train:", accuracy_score(y_train, y_pred_train))
print("Accuracy  test :", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix train:\n", confusion_matrix(y_train, y_pred_train))
print("Confusion Matrix test :\n", confusion_matrix(y_test, y_pred_test))

print("Classification Report:\n", classification_report(y_train, y_pred_train))
print("Classification Report:\n", classification_report(y_test, y_pred_test))





joblib.dump(best_model, "./models/best_knn_model.joblib")



Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time= 5.7min
[CV] END ...metric=euclidean, n_neighbors=5, weights=uniform; total time= 6.1min
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time= 6.2min
[CV] END ..metric=euclidean, n_neighbors=9, weights=distance; total time= 6.2min
[CV] END ..metric=euclidean, n_neighbors=11, weights=uniform; total time= 6.3min
[CV] END ..metric=euclidean, n_neighbors=9, weights=distance; total time= 6.4min
[CV] END ..metric=euclidean, n_neighbors=5, weights=distance; total time= 6.5min
[CV] END ..metric=euclidean, n_neighbors=5, weights=distance; total time= 6.5min
[CV] END ..metric=euclidean, n_neighbors=11, weights=uniform; total time= 6.6min
[CV] END ...metric=euclidean, n_neighbors=9, weights=uniform; total time= 6.6min
[CV] END ...metric=euclidean, n_neighbors=9, weights=uniform; total time= 6.6min
[CV] END ..metric=euclidean, n_neighbors=5, weig

ValueError: Found input variables with inconsistent numbers of samples: [7600, 120000]

In [ ]:
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)



if hasattr(best_model, "predict_proba"):
    y_prob_train = best_model.predict_proba(x_train )

    print("ROC AUC:",
          roc_auc_score(
              y_train,
              y_prob_train,
              multi_class='ovr'
          )
    )
    
if hasattr(best_model, "predict_proba"):
    y_prob_test = best_model.predict_proba(x_test)

    print("ROC AUC:",
          roc_auc_score(
              y_test,
              y_prob_test,
              multi_class='ovr'
          )
    )




print("Accuracy train:", accuracy_score(y_train, y_pred_train))
print("Accuracy  test :", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix train:\n", confusion_matrix(y_train, y_pred_train))
print("Confusion Matrix test :\n", confusion_matrix(y_test, y_pred_test))

print("Classification Report:\n", classification_report(y_train, y_pred_train))
print("Classification Report:\n", classification_report(y_test, y_pred_test))





ROC AUC: 0.9999999850925926
ROC AUC: 0.979237961680517
Accuracy train: 0.9998833333333333
Accuracy  test : 0.9122368421052631
Confusion Matrix train:
 [[29999     0     0     1]
 [    0 30000     0     0]
 [    2     0 29994     4]
 [    0     0     7 29993]]
Confusion Matrix test :
 [[1698   64   84   54]
 [  17 1858   19    6]
 [  51   18 1677  154]
 [  51   11  138 1700]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     30000
           1       1.00      1.00      1.00     30000
           2       1.00      1.00      1.00     30000
           3       1.00      1.00      1.00     30000

    accuracy                           1.00    120000
   macro avg       1.00      1.00      1.00    120000
weighted avg       1.00      1.00      1.00    120000

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.89      0.91      1900
           1       0.95      0